# W02｜Backend 分层 + Tool Calling｜教学版

**建议学习时间：4h**

这本 Notebook 的定位是“第二教材”，不是代码仓库。  
正确使用方式：

1. 先看课程视频；
2. 回到 Notebook 看“是什么 / 为什么 / 什么时候用”；
3. 带着“你应该观察什么”再运行代码；
4. 先自己做题，再展开参考答案；
5. 最后把同一机制接入 ChatBI。

## Excel 任务表 1:1 对照

| # | 模块 | Excel任务 | 输出/验收 | Notebook处理方式 |
|---:|---|---|---|---|
| 1 | Backend | 按下方课程映射完成 CRUD / 分层 / Tool Calling | 完成 Ch5/6/10 + Agent Under the Hood 指定小节 | 知识点 + Checkpoint/参考答案 |
| 2 | Agent | 学 Tool Calling / Tool Schema | 能解释模型如何选择工具 | 知识点 + Checkpoint/参考答案 |
| 3 | Lab | 把 Week1 endpoint 重构分层 | endpoint 不再塞业务逻辑 | 明确作业 + 参考答案/参考实现 |
| 4 | ChatBI | 加入 2 个 mock 查询工具 | Agent 能选择并调用正确工具 | 明确作业 + 参考答案/参考实现 |
| 5 | 复盘 | 写 3 条分层原则 | 能说明为什么这样拆 | 明确作业 + 参考答案/参考实现 |

> **规则**：Task 1-2 是学习任务，用 Checkpoint/面试题验收；Task 3-5 是作业任务，下面给明确参考答案/参考实现。设计题没有唯一答案时给参考图/评分标准。

## Excel 作业参考答案 / 验收标准

### Task 3｜分层重构参考
```text
router.py      -> HTTP / Pydantic / status code
service.py     -> business flow / orchestration
repository.py  -> data access
tools.py       -> Agent exposed capabilities
```
Endpoint 不应该直接写 SQL 或复杂 Agent orchestration。

### Task 4｜两个 mock tool
`get_metric_schema(metric)` + `run_mock_query(metric, dimension)`；模型只返回 tool name + arguments，程序验证并执行。

### Task 5｜三条分层原则参考答案
1. HTTP 与业务逻辑分离；
2. 数据访问通过 Repository/依赖注入隔离；
3. 每层只有一种主要变化原因，方便测试和替换。

> 建议：先完成作业，再对照参考答案。不要直接照抄。

## 本周学习目标

        - 理解 Router / Service / Repository 的职责
- 理解 Tool Calling 的真实执行链路
- 开始建立“模型决策、程序执行”的边界

        ## 本周前置检查

        - 完成 W01；会基本 Pydantic model 和 FastAPI endpoint

        ## 本周必须记住的英文术语

        - `service layer`：服务层
- `repository`：仓储/数据访问层
- `tool calling`：工具调用
- `tool schema`：工具结构
- `observation`：工具结果

        > 可以听中文翻译，但这些英文术语一定要保留，因为代码、日志、文档和面试都会使用英文。

## 1. 为什么不能把所有代码都写在 FastAPI endpoint 里？

    **是什么（What）**  
    一个 endpoint 如果同时处理 HTTP、业务规则、数据库、LLM、错误处理，会很快变成难测试、难修改的“大函数”。

    **为什么需要（Why）**  
    分层的目标不是“代码看起来高级”，而是降低耦合，让每层只承担一种变化。

    **什么时候用（When）**  
    Router 处理 HTTP；Service 处理业务流程；Repository 处理数据访问；Tool 暴露给 Agent。

    **常见误区 / 容易踩坑**  
    不要为了分层而分层。小 demo 不需要十几层，但 production code 应至少把 HTTP 与核心业务逻辑分开。


**中英术语**：`router` = 路由层；`service layer` = 业务服务层；`repository` = 数据访问层；`dependency injection` = 依赖注入

### 代码实验：同一个功能拆成 Repository 和 Service

**这段代码想让你看什么？**

观察：Service 不知道数据来自 dict、PostgreSQL 还是 API；以后换数据源时，Service 可以少改。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
class MetricRepository:
    def __init__(self):
        self._data = {
            "message_count": {"expression": "SUM(volume)", "description": "total messages"},
            "cost": {"expression": "SUM(cost)", "description": "total cost"},
        }

    def get(self, name: str):
        return self._data.get(name)

class MetricService:
    def __init__(self, repo: MetricRepository):
        self.repo = repo

    def explain_metric(self, name: str):
        item = self.repo.get(name)
        if not item:
            raise ValueError(f"Unknown metric: {name}")
        return item

service = MetricService(MetricRepository())
print(service.explain_metric("message_count"))

### Checkpoint A

1. 如果数据库从 PostgreSQL 换成远程 API，哪一层最应该变化？
2. Router 是否应该直接写 SQL？为什么？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** Repository / data access 层最应该变化，Service 尽量保持业务语义不变。

**2.** 一般不应该。这样 HTTP 层会和数据访问强耦合，也更难测试和复用。

    </details>

## 2. Tool Calling 本质是什么？

    **是什么（What）**  
    模型不是直接执行 Python 函数，而是先生成“我要调用哪个工具、参数是什么”的结构化请求，然后你的程序验证并执行。

    **为什么需要（Why）**  
    这样模型只能使用你显式允许的能力，并能通过 schema 限制参数。

    **什么时候用（When）**  
    Agent 查询指标、检索 metadata、调用 MCP、执行安全分析动作。

    **常见误区 / 容易踩坑**  
    Tool 描述越多越好是错的；工具太相似会让模型选错。更危险的是把任意 SQL/任意 Python 暴露成 Tool。


**中英术语**：`tool schema` = 工具参数结构；`tool call` = 工具调用请求；`observation` = 工具执行结果；`agent loop` = 智能体循环

### 代码实验：自己实现一个最小 Tool Registry

**这段代码想让你看什么？**

观察：schema 在函数执行前做验证；model 只决定 tool name + arguments，真正执行权在程序。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
from pydantic import BaseModel

class MetricArgs(BaseModel):
    metric: str

class QueryArgs(BaseModel):
    metric: str
    dimension: str

def get_metric_schema(metric: str):
    return service.explain_metric(metric)

def run_mock_query(metric: str, dimension: str):
    return {"metric": metric, "dimension": dimension, "rows": 12}

TOOLS = {
    "get_metric_schema": (MetricArgs, get_metric_schema),
    "run_mock_query": (QueryArgs, run_mock_query),
}

def execute_tool(name: str, arguments: dict):
    if name not in TOOLS:
        raise ValueError(f"Unknown tool: {name}")
    schema, fn = TOOLS[name]
    parsed = schema(**arguments)
    return fn(**parsed.model_dump())

print(execute_tool("get_metric_schema", {"metric": "message_count"}))

### 代码实验：看懂最小 Agent Loop

**这段代码想让你看什么？**

观察循环的四步：Question → Model Decision → Tool Execution → Observation。LangChain/LangGraph 本质上是在帮你管理这种循环和状态。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
def mock_model(question: str):
    if "definition" in question.lower():
        return {
            "type": "tool",
            "name": "get_metric_schema",
            "arguments": {"metric": "message_count"},
        }
    return {
        "type": "tool",
        "name": "run_mock_query",
        "arguments": {"metric": "message_count", "dimension": "channel"},
    }

question = "Compare message volume by channel"
decision = mock_model(question)
observation = execute_tool(decision["name"], decision["arguments"])

print("Question:", question)
print("Model decision:", decision)
print("Tool observation:", observation)

### 练习：新增一个 Tool

增加 `list_available_metrics`，不需要参数，返回可用 metric 列表。

<details>
<summary><b>提示</b></summary>

无参数 tool 可以用一个空 Pydantic model。

</details>

In [ ]:
# TODO: 自己先写 list_available_metrics + 注册到 TOOLS

<details>
    <summary><b>参考答案</b></summary>

    ```python
class EmptyArgs(BaseModel):
    pass

def list_available_metrics():
    return ["message_count", "cost"]

TOOLS["list_available_metrics"] = (EmptyArgs, lambda: list_available_metrics())
```

    </details>

### 练习：错误分类

以下错误哪些适合 retry：Tool 名不存在、HTTP Tool 返回 503、metric 不存在、参数 schema 错误？

<details>
    <summary><b>参考答案</b></summary>

    ```python
# Tool 名不存在 -> 不重试，程序/模型决策错误
# 503 -> 可能重试，属于临时下游不可用
# metric 不存在 -> 不重试，业务输入/语义错误
# schema 错误 -> 不重试，调用参数错误
```

    </details>

## ChatBI 本周作业

        把 W01 的 `/api/query` 重构成：
`Router → QueryService → Tool Registry`

至少实现两个工具，并写 2 个 API test。

### 本周面试 / 理解检查

1. Service Layer 解决的核心问题是什么？
2. Tool Calling 为什么比让 LLM 直接执行任意代码安全？
3. 什么错误可以 retry？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 把业务流程从 HTTP/DB 等基础设施细节中分离出来，便于测试、复用和替换依赖。

**2.** 因为程序控制工具集合、schema、权限和真实执行过程，模型只有受限决策权。

**3.** 通常只有临时性下游错误，如 timeout/429/503；参数错误、权限错误、业务校验失败不应盲目重试。

    </details>

## 本周结束前的自检

只有下面 4 项都能做到，才算真正完成：

- [ ] 我能不用课程原话，用自己的话解释核心概念；
- [ ] 我知道这周的代码实验到底在证明什么；
- [ ] 我至少独立完成了一道代码题；
- [ ] 我能说出一个“生产环境里如果做错会发生什么”的例子。